# 06 — Grad-CAM analysis (pretrained ResNet-18)

Advanced direction (spec): visualise which regions drive predictions, compare
**correct vs incorrect** cases, then inspect **confusable pairs** from the
confusion matrix. Maps alone are not enough — each figure needs a short
interpretation (organism vs background/context).

**Citation:** Selvaraju et al., Grad-CAM, ICCV 2017.  
**Code:** `pytorch-grad-cam` via `src/gradcam_utils.py`.

**Prerequisite:** `checkpoints/resnet18_pretrained/best.pt` (or Drive copy).


## 0. Colab bootstrap


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"IN_COLAB = {IN_COLAB}")

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_SUBSET = Path("/content/drive/MyDrive/comp9517/subset")
    REPO_DIR = Path("/content/Haramcomp9517")
    os.environ["INAT_DATA_ROOT"] = str(DRIVE_SUBSET)
    if not REPO_DIR.is_dir():
        try:
            subprocess.check_call(
                ["git", "clone", "-b", "nate/pretrained-gradcam", "--single-branch", "https://github.com/NateRebello/Haramcomp9517.git", str(REPO_DIR)]
            )
        except subprocess.CalledProcessError:
            subprocess.check_call(["git", "clone", "https://github.com/NateRebello/Haramcomp9517.git", str(REPO_DIR)])
    pipeline = REPO_DIR / "dl_pipeline"
    os.chdir(pipeline)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "grad-cam"])
    if str(pipeline) not in sys.path:
        sys.path.insert(0, str(pipeline))
    print("INAT_DATA_ROOT =", os.environ["INAT_DATA_ROOT"])
else:
    print("Local — ensure grad-cam is installed: pip install grad-cam")


## 1. Setup


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

PIPELINE_ROOT = Path.cwd().resolve()
if PIPELINE_ROOT.name == "notebooks":
    PIPELINE_ROOT = PIPELINE_ROOT.parent
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from src.config import CHECKPOINTS_DIR, NUM_CLASSES, RESULTS_DIR, SEED, ensure_output_dirs, set_seed
from src.dataset import build_dataloaders, build_datasets, idx_to_category_id
from src.gradcam_utils import (
    compute_cam_overlay,
    make_gradcam,
    predict_topk,
    sample_indices,
    split_correct_incorrect,
    tensor_to_uint8_rgb,
)
from src.metrics import collect_predictions, make_confusion_matrix, most_confused_pairs
from src.models import build_model
from src.train_utils import load_checkpoint

set_seed(SEED)
ensure_output_dirs()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = RESULTS_DIR / "gradcam_pretrained"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("device=", device, "out=", OUT_DIR)


## 2. Load pretrained checkpoint + test set


In [ ]:
CKPT = CHECKPOINTS_DIR / "resnet18_pretrained" / "best.pt"
if not CKPT.is_file():
    CKPT = Path("/content/drive/MyDrive/comp9517/checkpoints/resnet18_pretrained/best.pt")
assert CKPT.is_file(), f"Missing pretrained checkpoint: {CKPT}"

model = build_model("resnet18", num_classes=NUM_CLASSES, pretrained=False)
load_checkpoint(CKPT, model, map_location=device)
model.to(device).eval()

train_ds, val_ds, test_ds = build_datasets(augment_train=False)
_, _, test_loader = build_dataloaders(train_ds, val_ds, test_ds, batch_size=32)
cat_id = idx_to_category_id(train_ds)
print("loaded", CKPT, "test images", len(test_ds))


## 3. Test predictions (for correct/incorrect split)


In [ ]:
preds = collect_predictions(model, test_loader, device, use_amp=(device.type == "cuda"), topk=5)
y_true, y_pred = preds["y_true"], preds["y_pred"]
correct_idx, incorrect_idx = split_correct_incorrect(y_true, y_pred)
print(f"correct={len(correct_idx)}  incorrect={len(incorrect_idx)}  "
      f"top1={ (y_true==y_pred).mean()*100:.2f}%")

cm = make_confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
confused = most_confused_pairs(cm, cat_id, top_n=10)
print("Top confused pairs:")
for a, b, c in confused:
    print(f"  {a} → {b}  (n={c})")


## 4. Grad-CAM: correct vs incorrect

For each example we show the RGB image and the Grad-CAM overlay for the
**predicted** class. While viewing, ask: does the heat sit on the organism,
or on background/context (leaves, sky, feeder, watermark)?


In [ ]:
cam = make_gradcam(model)
rng = np.random.default_rng(SEED)
n_show = 4
sel_correct = sample_indices(correct_idx, n_show, rng)
sel_incorrect = sample_indices(incorrect_idx, n_show, rng)

# Materialise test samples by index (ImageFolder supports __getitem__)
def show_cam_panel(indices, title_prefix: str, save_name: str):
    fig, axes = plt.subplots(len(indices), 2, figsize=(6, 3 * len(indices)))
    if len(indices) == 1:
        axes = np.array([axes])
    notes = []
    for row, idx in enumerate(indices):
        img, y = test_ds[idx]
        x = img.unsqueeze(0).to(device)
        with torch.no_grad():
            pred = model(x).argmax(dim=1).item()
        gray, overlay = compute_cam_overlay(cam, x, target_category=pred)
        rgb = tensor_to_uint8_rgb(img)
        axes[row, 0].imshow(rgb)
        axes[row, 0].set_title(
            f"true={cat_id[y]}  pred={cat_id[pred]}", fontsize=9
        )
        axes[row, 0].axis("off")
        axes[row, 1].imshow(overlay)
        axes[row, 1].set_title("Grad-CAM (pred class)", fontsize=9)
        axes[row, 1].axis("off")
        ok = "CORRECT" if y == pred else "WRONG"
        notes.append(
            f"{ok}: true {cat_id[y]} pred {cat_id[pred]} — "
            f"inspect whether activation covers the organism vs background."
        )
    fig.suptitle(title_prefix, fontsize=12)
    fig.tight_layout()
    path = OUT_DIR / save_name
    fig.savefig(path, dpi=150)
    plt.show()
    print("saved", path)
    for n in notes:
        print("-", n)

show_cam_panel(sel_correct, "Correct predictions", "gradcam_correct.png")
show_cam_panel(sel_incorrect, "Incorrect predictions", "gradcam_incorrect.png")


## 5. Confusable species pair

Take the top confused pair from the matrix. Show a few images from each true
class and Grad-CAM for the (wrong) predicted class — does the model fire on
shared traits (pose, colour patch, habitat) rather than species-specific cues?


In [ ]:
if not confused:
    print("No off-diagonal confusions — skip pair analysis.")
else:
    true_name, pred_name, cnt = confused[0]
    true_i = train_ds.class_to_idx[true_name]
    pred_i = train_ds.class_to_idx[pred_name]
    print(f"Focus pair: true={true_name} (idx {true_i}) → pred={pred_name} (idx {pred_i}), count={cnt}")

    # Find test indices where true=true_i and pred=pred_i
    pair_idxs = np.where((y_true == true_i) & (y_pred == pred_i))[0]
    pair_idxs = sample_indices(pair_idxs, min(4, len(pair_idxs)), rng)
    if not pair_idxs:
        print("No examples for this ordered pair in the sampled predictions.")
    else:
        show_cam_panel(
            pair_idxs,
            f"Confusions: {true_name} predicted as {pred_name}",
            "gradcam_confused_pair.png",
        )
        print(
            "Interpretation prompt: Do CAMs highlight the same body part / background "
            "context on both species? That suggests the model relies on shared cues."
        )


## 6. Write a short analysis note (for the report)

Edit the string below after looking at the figures — this is what earns credit
beyond pretty heatmaps.


In [ ]:
analysis = {
    "checkpoint": str(CKPT),
    "n_correct": int(len(correct_idx)),
    "n_incorrect": int(len(incorrect_idx)),
    "top_confused_pairs": [
        {"true": a, "pred": b, "count": c} for a, b, c in confused
    ],
    "findings": [
        "TODO: On correct examples, Grad-CAM typically focused on … (organism body / head / flower).",
        "TODO: On incorrect examples, activation often spilled onto … (background / similar co-occurring object).",
        "TODO: For the top confused pair, both species share … which may explain the swap.",
    ],
}
out = OUT_DIR / "gradcam_analysis_notes.json"
with out.open("w", encoding="utf-8") as f:
    json.dump(analysis, f, indent=2)
print("Edit findings in", out, "after inspecting the PNGs.")


## 7. Done

Artefacts under `results/gradcam_pretrained/`:
- `gradcam_correct.png` / `gradcam_incorrect.png` / `gradcam_confused_pair.png`
- `gradcam_analysis_notes.json` — fill in the TODOs for the report / video
